In [4]:
import torch
import torch.nn as nn 
import tiktoken 

In [8]:
class MultiHeadAttention(nn.Module): 
  def __init__(self, d_in, d_out, context_length, drop_rate, n_heads, qkv_bias=False): 
    super().__init__()
    self.d_in = d_in
    self.d_out = d_out
    self.context_length = context_length
    self.drop_rate = drop_rate
    self.n_heads = n_heads
    self.head_dim = d_out // n_heads 
    assert d_out % n_heads == 0 

    # weight matrices: (8, 16), PyTorch converts to (16, 8) by default
    self.W_Q = nn.Linear(d_in, d_out, bias=qkv_bias)                    
    self.W_K = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_V = nn.Linear(d_in, d_out, bias=qkv_bias)

    # mask fill 
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    # dropout 
    self.dropout = nn.Dropout(drop_rate)

    # output projection 
    self.out_proj = nn.Linear(d_out, d_out)
  
  def forward(self, x):
    batch, num_tokens, d_in = x.shape 

    # reshape qkv matrices
    queries = self.W_Q(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)
    keys = self.W_K(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)
    values = self.W_V(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)

    print("queries:", queries.shape)
    print("keys:", keys.shape)
    print("values:", values.shape)
    
    # attention score 
    attn_score = queries @ keys.transpose(-2, -1)
    scaled_score = attn_score / (self.head_dim ** 0.5)

    # applying causal attention with attention mask 
    scaled_score = scaled_score.masked_fill(self.mask[:num_tokens, :num_tokens], float("-inf"))

    attn_weight = torch.softmax(scaled_score, dim=-1)
    attn_weight = self.dropout(attn_weight)

    # return context vector
    context_vec = (attn_weight @ values).transpose(1,2).contiguous().view(batch, num_tokens, self.d_out)

    return self.out_proj(context_vec) 

In [10]:
torch.manual_seed(420)
x = torch.randn(2, 4, 8)          # b=2, num_tokens=4, d_in=8
attn = MultiHeadAttention(d_in=8, d_out=16, context_length=1024, n_heads=4, drop_rate=0.1)
out = attn(x)
out.shape 


queries: torch.Size([2, 4, 4, 4])
keys: torch.Size([2, 4, 4, 4])
values: torch.Size([2, 4, 4, 4])


torch.Size([2, 4, 16])

In [ ]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))
    self.eps = 1e-5

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False) 
    norm_x = (x - mean) / torch.sqrt(var + self.eps)
    return self.scale * norm_x + self.shift 

class FeedForward(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Linear(emb_dim, 4 * emb_dim), 
      nn.GELU(approximate='tanh'),
      nn.Linear(emb_dim * 4, emb_dim), 
    )
  
  def forward(self, x):
    return self.layers(x)

In [ ]:
from torch.nn import Dropout


class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.norm1 = LayerNorm(cfg["emb_dim"])
    self.attn = MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      drop_rate=cfg["drop_rate"],
      n_heads=cfg["n_heads"],
      qkv_bias=cfg["qkv_bias"]
    )
    self.ff = FeedForward(cfg["emb_dim"])
    self.norm2 = LayerNorm(cfg["emb_dim"])
    self.dropout = Dropout(cfg["drop_rate"])

  def forward(self, x):
    # sub-layer 1
    shortcut = x
    x = self.norm1(x)
    x = self.attn(x)
    x = self.dropout(x)
    x = x + shortcut

    # sub-layer 2
    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x = self.dropout(x)
    x = x + shortcut

    return x

In [ ]:
GPT3_CONFIG_125M = {
    "vocab_size": 50257,
    "context_length": 2048,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

In [14]:
class GPT(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.vec_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.dropout = nn.Dropout(cfg["drop_rate"])
    self.layers = nn.Sequential(
      # *, unpacks the list into individual arguments
      *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )
    self.lnf = LayerNorm(cfg["emb_dim"])
    self.lm_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

  def forward(self, x):
    batch_size, seq_len = x.shape
    pos = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
    
    x = self.vec_emb(x) + self.pos_emb(pos)
    x = self.dropout(x)
    x = self.layers(x)
    x = self.lnf(x)
    logits = self.lm_head(x)
    return logits



In [15]:
torch.manual_seed(123)
model = GPT(GPT3_CONFIG_125M)
model.eval()

# fake token indices — integers in range [0, vocab_size)
x = torch.randint(0, GPT3_CONFIG_125M["vocab_size"], (2, 10))
logits = model(x)
print(logits.shape)

NameError: name 'GPT3_CONFIG_125M' is not defined